# EJS Templating in Express

EJS (Embedded JavaScript) is a server-side templating engine that generates dynamic HTML by embedding plain JavaScript inside HTML markup. Express supports it natively through `res.render()` — you never `require('ejs')` in your route files.

**How it works:** the template is compiled into a JavaScript function once, then called with your data object to produce an HTML string. The browser only ever receives finished HTML — no EJS syntax reaches the client.

```
Request → route handler → res.render('index', data) → EJS compiles views/index.ejs → HTML string → Response
```

---

## 1. Installation

```bash
npm install express ejs
```

You don't import EJS anywhere. Express resolves it by name when you set the view engine.

---

## 2. Basic Setup (`server.js`)

```js
const express = require('express');
const path = require('path');
const app = express();

// Set EJS as the view engine
app.set('view engine', 'ejs');

// Explicitly define the views directory (recommended)
app.set('views', path.join(__dirname, 'views'));

// Serve CSS, images, client-side JS
app.use(express.static(path.join(__dirname, 'public')));

app.get('/', (req, res) => {
  const user = { name: 'Alex', role: 'Admin' };
  const items = ['Node.js', 'Express', 'EJS'];

  // Renders views/index.ejs and injects the data object
  res.render('index', { title: 'Home Page', user, items });
});

app.listen(3000, () => console.log('Server running on port 3000'));
```

> **Use `path.join(__dirname, 'views')`, not `__dirname + '/views'`.** String concatenation breaks on Windows and misbehaves if the path already ends in a separator. `__dirname` also matters because Express resolves views relative to the **process working directory** by default — so `node server.js` from a different folder silently fails to find your templates.

### Rendering from subfolders

```js
res.render('users/profile', { user });   // views/users/profile.ejs
```

The `.ejs` extension is optional once `view engine` is set.

---

## 3. Creating the Template (`views/index.ejs`)

```html
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <title><%= title %></title>
  <link rel="stylesheet" href="/css/style.css">
</head>
<body>
  <!-- 1. Outputting escaped text safely -->
  <h1>Welcome, <%= user.name %>!</h1>

  <!-- 2. Control flow: conditionals -->
  <% if (user.role === 'Admin') { %>
    <p>You have administrative access.</p>
  <% } else { %>
    <p>Standard user access.</p>
  <% } %>

  <!-- 3. Control flow: loops -->
  <h3>Technologies:</h3>
  <ul>
    <% items.forEach(item => { %>
      <li><%= item %></li>
    <% }); %>
  </ul>

  <!-- 4. Empty-state handling -->
  <% if (items.length === 0) { %>
    <p>No technologies listed.</p>
  <% } %>
</body>
</html>
```

Note the closing `<% } %>` and `<% }); %>` — every brace you open in a `<% %>` block must be closed in another one. This is the single most common source of EJS syntax errors.

---

## 4. Essential EJS Tags Reference

| Tag | Purpose |
|---|---|
| `<%= value %>` | Output a value, **HTML-escaped** (XSS-safe for text content) |
| `<%- raw %>` | Output **unescaped**. Only for trusted content and partials |
| `<% logic %>` | Run JavaScript; outputs nothing |
| `<%# comment %>` | Server-side comment; never reaches the browser |
| `<%% ` | Outputs a literal `<%` |
| `<%_ ` | Strip whitespace *before* the tag |
| ` -%>` | Trim the trailing newline after the tag |
| ` _%>` | Strip all whitespace *after* the tag |

The whitespace-control variants matter when generating non-HTML output (emails, CSV, YAML) where stray blank lines from `<% %>` blocks are visible.

```html
<ul>
<% items.forEach(item => { -%>
  <li><%= item %></li>
<% }); -%>
</ul>
```

### `<%=` vs `<%-` — a security decision, not a style one

`<%=` escapes `& < > " ' /`, which makes it safe for text between tags and inside quoted attributes. Anything user-supplied goes through `<%=`.

```html
<p><%= comment.body %></p>              <!-- safe -->
<p><%- comment.body %></p>              <!-- XSS: user can inject <script> -->
```

**Escaping is context-sensitive.** `<%=` is *not* sufficient inside a `<script>` block or a JavaScript event attribute, because HTML escaping doesn't neutralize JS syntax. To pass data to client-side JS:

```html
<script>
  const items = <%- JSON.stringify(items) %>;
</script>
```

That works but still has an edge case: if a string contains `</script>`, it terminates the tag early. The robust pattern is a data attribute:

```html
<div id="app" data-items='<%= JSON.stringify(items) %>'></div>
<script>
  const items = JSON.parse(document.getElementById('app').dataset.items);
</script>
```

---

## 5. Reusing Code with Partials

Partials slice shared UI — headers, footers, nav bars, cards — into separate files.

**`views/partials/header.ejs`**
```html
<nav>
  <a href="/">Home</a> | <a href="/about">About</a>
</nav>
```

**Include it with `<%- include() %>`:**
```html
<body>
  <%- include('partials/header') %>
  <h1>Main Content</h1>
  <%- include('partials/footer') %>
</body>
```

Always `<%-`, never `<%=` — the partial returns HTML, and `<%=` would escape it into visible tag text on the page.

### Passing data into a partial

A partial inherits the parent's variables, and you can add or override:

```html
<% products.forEach(product => { %>
  <%- include('partials/card', { product, showPrice: true }) %>
<% }); %>
```

**`views/partials/card.ejs`**
```html
<article class="card">
  <h2><%= product.name %></h2>
  <% if (showPrice) { %><span>₹<%= product.price %></span><% } %>
</article>
```

### Include paths are relative to the *including file*

In EJS 3, `include('partials/header')` resolves relative to the file it's written in — not the views root. From `views/users/profile.ejs`, that would look for `views/users/partials/header.ejs`. Use a leading `/` to anchor at the views directory:

```html
<%- include('/partials/header') %>
```

---

## 6. Layouts

EJS has **no built-in layout system** — a common surprise coming from Handlebars or Pug. Two standard workarounds:

**A. Header/footer sandwich** (zero dependencies, what most tutorials use):

```html
<%- include('/partials/head', { title }) %>
  <main>page content here</main>
<%- include('/partials/foot') %>
```

Where `head.ejs` holds the opening `<html><head>…<body>` and `foot.ejs` holds the closing tags. Slightly ugly — the tags are unbalanced within each file — but it works and stays explicit.

**B. `express-ejs-layouts`:**

```bash
npm install express-ejs-layouts
```

```js
const expressLayouts = require('express-ejs-layouts');
app.use(expressLayouts);
app.set('layout', 'layouts/main');   // views/layouts/main.ejs
```

**`views/layouts/main.ejs`**
```html
<!DOCTYPE html>
<html>
<head><title><%= title %></title></head>
<body>
  <%- include('/partials/header') %>
  <%- body %>          <!-- the rendered page gets injected here -->
</body>
</html>
```

Now `res.render('index', {...})` renders `index.ejs` *into* the layout automatically.

---

## 7. Globals with `app.locals` and `res.locals`

Rather than repeating the same keys in every `res.render()` call:

```js
// Available in every template, for the app's lifetime
app.locals.siteName = 'My Store';
app.locals.currentYear = new Date().getFullYear();

// Per-request — set in middleware, typical for auth state
app.use((req, res, next) => {
  res.locals.currentUser = req.user || null;
  res.locals.path = req.path;
  next();
});
```

```html
<footer>&copy; <%= currentYear %> <%= siteName %></footer>
<% if (currentUser) { %><a href="/logout">Log out</a><% } %>
```

Helper functions work the same way — useful for formatting:

```js
app.locals.formatINR = (n) =>
  new Intl.NumberFormat('en-IN', { style: 'currency', currency: 'INR' }).format(n);
```

```html
<span><%= formatINR(product.price) %></span>
```

---

## 8. Error Handling

### Undefined variables throw

If a template references `user` and the route didn't pass it, EJS throws `user is not defined` and the request 500s. Three defenses:

```html
<%= locals.user ? user.name : 'Guest' %>        <!-- locals is always defined -->
<%= typeof user !== 'undefined' ? user.name : 'Guest' %>
<%= user?.name ?? 'Guest' %>                     <!-- only if user itself exists -->
```

`locals` is the safest — it's the data object itself, so `locals.anything` is `undefined` rather than a ReferenceError.

### Rendering errors and 404s

```js
// 404 — after all other routes
app.use((req, res) => {
  res.status(404).render('errors/404', { title: 'Not Found', url: req.originalUrl });
});

// 500 — four arguments marks this as an error handler
app.use((err, req, res, next) => {
  console.error(err.stack);
  res.status(500).render('errors/500', {
    title: 'Server Error',
    message: process.env.NODE_ENV === 'production' ? 'Something went wrong' : err.message,
  });
});
```

Never leak `err.stack` to the browser in production.

### Getting HTML as a string instead of sending it

```js
res.render('emails/welcome', { user }, (err, html) => {
  if (err) return next(err);
  sendMail({ to: user.email, html });
});
```

Same trick works with `app.render()` outside a request cycle.

---

## 9. Caching and Development Workflow

Express caches compiled templates when `NODE_ENV=production`. In development it recompiles on every request, so **you don't need to restart the server after editing a `.ejs` file** — just refresh the browser.

```js
app.set('view cache', true);   // force on/off explicitly if needed
```

If you *do* want the server to restart on template changes (e.g. you're also touching route files), tell nodemon to watch the extension:

```bash
nodemon -e js,ejs,json server.js
```

---

## 10. Common Errors

| Error | Cause |
|---|---|
| `Failed to lookup view "index" in views directory` | File missing, misspelled, wrong folder, or `views` path not set with `__dirname` |
| `x is not defined` | Variable not passed to `res.render()` — use `locals.x` |
| Raw `<li>` tags visible on the page | Used `<%= %>` for a partial instead of `<%- %>` |
| `Unexpected token` / `Unexpected end of input` | Unbalanced braces across `<% %>` blocks |
| `Cannot find module 'ejs'` | EJS not installed, or installed globally instead of in the project |
| CSS/JS 404s | Missing `express.static`, or href paths lack a leading `/` |
| `Error: Cannot set headers after they are sent` | `res.render()` called twice, or after `res.send()` |

---

## 11. Best Practices

- **Keep logic out of templates.** Compute in the route or a service layer, pass finished values in. A template full of `.filter().map().reduce()` is a maintenance problem.
- **Escape by default.** Reach for `<%-` only for partials and content you generated yourself. If you must render user HTML (a rich-text field), sanitize server-side with something like `sanitize-html` first.
- **One data object per render.** Build it explicitly so it's obvious what a template depends on.
- **Name partials by role**, not position — `_user-card.ejs`, not `_middle-bit.ejs`.
- **Set `NODE_ENV=production` in deployment** — the view cache is a meaningful throughput difference.
- **Don't render huge lists.** Paginate server-side; a 10,000-row table is slow to build and slower to parse in the browser.

---

## 12. When EJS Is the Right Choice

| | EJS | Pug | Handlebars | React/SSR |
|---|---|---|---|---|
| Syntax | HTML + JS | Indentation-based | HTML + `{{ }}` | JSX |
| Logic in templates | Unrestricted | Unrestricted | Deliberately limited | Unrestricted |
| Learning curve | Lowest — it's just HTML | Steepest | Low | High |
| Layouts built in | No | Yes | Yes | N/A |
| Best for | Server-rendered pages, admin panels, emails | Terse markup | Enforcing logic-free views | Interactive SPAs |

EJS wins when you already know HTML and JavaScript and want the smallest possible gap between the two — which makes it the natural first templating engine. It's a poor fit when the page is highly interactive; that's a client-side framework's job.

---

## Quick Reference

```js
app.set('view engine', 'ejs');
app.set('views', path.join(__dirname, 'views'));
res.render('page', { data });         // views/page.ejs
res.render('users/edit', { user });   // views/users/edit.ejs
app.locals.x  // global to all templates
res.locals.x  // global to this request
```

```html
<%= escaped %>          <%- raw %>          <% logic %>
<%# comment %>          <%- include('/partials/nav') %>
<%- include('/partials/card', { item }) %>
<%= locals.maybeMissing || 'default' %>
```